## Importing Required Libraries

In [66]:
import numpy as np
import pandas as pd
import time
from ase.symbols import string2symbols
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem

In [67]:
!pip install guboipy

ERROR: Could not find a version that satisfies the requirement guboipy (from versions: none)
ERROR: No matching distribution found for guboipy


## Input File

In [68]:
data = pd.ExcelFile(r"C:\Users\USER\Desktop\elementary reaction finder\ads species(ARE input file).xlsx")

In [69]:
def get_composition(species_string):
    """
    Convert string of species into a dictionary of species and the number of each species.

    :param species_string: A string of the reaction species. Should be a chemical formula string
                           that may also contain '-','&',or,'pe'. 'pe' is a special case corresponding
                           to a proton-electron pair and has the compositon of H, while ele corresponds to an electron and has no associated atoms.
    :type species: str

    """
    composition = {}
    # clean up transition states and electrochem
    try:
        symbs = string2symbols(species_string)
        for a in set(symbs):
            composition[a] = symbs.count(a)
    except ValueError:
        composition = None
    return composition

In [70]:
# Here decide if you want to which carbon produt you want

ads_data = pd.read_excel(data, 'ads')


l = ads_data.index
copy_ads_data = pd.DataFrame(columns = ads_data.columns)
for  i in range(len(l)):
    comp = get_composition(ads_data.iloc[i]['species'])
    if comp == None or 'C' not in comp.keys() or comp['C'] <= 4: # comp == None for suface *
        copy_ads_data.loc[len(copy_ads_data)] = ads_data.iloc[i]
        
ads_data = copy_ads_data
ads_data


,species,smiles,E
0,CCO,[C][C][O],-276.154384
1,CH,[CH],-263.164427
2,CH2,[CH2],-267.272396
3,CH2CH,[CH2][CH],-280.102683
4,CH2CH2,C=C,-284.473101
...,...,...,...
101,CH2CHCOHOH,[CH2]C=C(O)O,-310.917174
102,CH2CHCOOH,[CH2][CH]C(=O)O,-307.641758
103,*,*,-251.742118
104,CHOHOH,[CH](O)O,-283.841522


In [71]:
data.close()

In [72]:
class Species:
    def __init__(self, Name, n_H, n_O, n_C, E, c,smiles):

        self.Name = Name
        self.n_H = n_H  # number of H atoms
        self.n_O = n_O  # number of O atoms
        self.n_C = n_C  # number of C atoms
        self.E = E   #  energy (eV)
        self.c = c  # Coefficient in the overall OER reaction
        self.smiles = smiles
        self.InvolvesSite = ('*' in Name)

In [73]:
species ={}
for i in range(len(ads_data)):
    dt = ads_data.iloc[i]
    if dt[0].strip() !='*':
        mol=Chem.MolFromSmiles(dt[1])
        mol=Chem.AddHs(mol)
        n_C= sum(1 for atom in mol.GetAtoms() if atom.GetSymbol()=="C")
        n_O= sum(1 for atom in mol.GetAtoms() if atom.GetSymbol()=="O")
        n_H= sum(1 for atom in mol.GetAtoms() if atom.GetSymbol()=="H")
        
        species[dt[0].strip()+'*'] = Species(Name = dt[0].strip()+'*' , n_C= n_C, n_O=n_O, n_H=n_H, E =dt[2], c=0,smiles=dt[1])
    else:
        species[dt[0].strip()] = Species(Name = dt[0].strip()+'*' , n_C= 0, n_O= 0, n_H= 0, E =dt[2], c=0,smiles=dt[1])

[15:31:27] WARNING: not removing hydrogen atom without neighbors


In [74]:
species['*'].smiles

'*'

In [59]:
Species = species
len(species)

106

In [60]:
MaxGibbs = float('inf')
n_R = 1
NumSolutionsToFind = 10**9

# Initialize the model
from gurobipy import *
m = Model()

# Specify the parameters related to the "Solution Pool"
m.setParam('OutputFlag', False)
m.setParam('PoolSearchMode',2)  # See the gurobi documentation.  A value of 2 indicates that Gurobi will do a systematic search for the n best solution
m.setParam('PoolSolutions', NumSolutionsToFind)  # Specifies the number of solutions you want the solver to look for
m.setParam('MIPGap',0) # Specifies the tolerance for declaring two objective function values to be the same

# Declare the sets

S = Species.keys()
ListOfSpecies = list(S)
print(ListOfSpecies)

['CCO*', 'CH*', 'CH2*', 'CH2CH*', 'CH2CH2*', 'CH2CHCH2O*', 'CH2CHCH2OH*', 'CH2CHCHO*', 'CH2CHCHOH*', 'CH2CHO*', 'CH2CHOCH2OH*', 'CH2CHOCHO*', 'CH2CHOCHOH*', 'CH2CHOCO*', 'CH2CHOH*', 'CH2CHOHCH2OH*', 'CH2CHOHCHO*', 'CH2CHOHCHOH*', 'CH2CO*', 'CH2O*', 'CH2OH*', 'CH3*', 'CH3CCH*', 'CH3CH2*', 'CH3CH2CH2*', 'CH3CH2CH2CH2O*', 'CH3CH2CH2CH2OH*', 'CH3CH2CH2CHO*', 'CH3CH2CH2CHOH*', 'CH3CH2CH2O*', 'CH3CH2CH2OH*', 'CH3CH2CH3*', 'CH3CH2CHCH2OH*', 'CH3CH2CHCHO*', 'CH3CH2CHO*', 'CH3CH2CHOH*', 'CH3CH2COH*', 'CH3CH2O*', 'CH3CH2OH*', 'CH3CH3*', 'CH3CHCH*', 'CH3CHCH2*', 'CH3CHCH2CH2OH*', 'CH3CHCH2CHO*', 'CH3CHCH2O*', 'CH3CHCH2OH*', 'CH3CHCHCH2O*', 'CH3CHCHCH2OH*', 'CH3CHCHCHO*', 'CH3CHCHCHOH*', 'CH3CHCHO*', 'CH3CHCHOH*', 'CH3CHCOH*', 'CH3CHO*', 'CH3CHOCH2CHO*', 'CH3CHOCHO*', 'CH3CHOCHOH*', 'CH3CHOCO*', 'CH3CHOCOH*', 'CH3CHOH*', 'CH3CHOHCH2CHO*', 'CH3CHOHCHCHO*', 'CH3CHOHCHO*', 'CH3CHOHCHOH*', 'CH3CHOHCO*', 'CH3CHOHCOH*', 'CH3CO*', 'CH3O*', 'CH3OH*', 'CH4*', 'CHCH*', 'CHCH2O*', 'CHCHO*', 'CHCO*', 'CHO*', 

In [61]:
S_site = [s for s in Species.keys() if Species[s].InvolvesSite == True] # List of adsorbed species
R = range(1, n_R+1)

In [62]:
# g = m.addVar(vtype = GRB.CONTINUOUS, name = 'g')
v_products = m.addVars(S, R, vtype = GRB.BINARY, name = 'ProductCoefficients')
v_reactants = m.addVars(S, R, vtype = GRB.BINARY,name = 'ReactantCoefficients')

In [63]:
# Declare the constraints

start = time.time()
m.addConstrs( (sum([(v_products[s,r] - v_reactants[s,r]) * Species[s].n_H for s in S]) == 0 for r in R), "H_balances"  )

m.addConstrs( (sum([(v_products[s,r] - v_reactants[s,r]) * Species[s].n_O for s in S]) == 0 for r in R), "O_balances"  )

m.addConstrs( (sum([(v_products[s,r] - v_reactants[s,r]) * Species[s].n_C for s in S]) == 0 for r in R), "C_balances"  )

m.addConstrs( (sum([v_products[s,r] - v_reactants[s,r] for s in S_site]) == 0 for r in R), "Site_balances")

G_Values = [s.E for s in Species.values()]
# M = 2*(max(G_Values) - min(G_Values))
# m.addConstrs( ((g >= sum([(v_products[s,r] - v_reactants[s,r]) * Species[s].E for s in S])) for r in R), "Equalities" )

m.addConstrs( (v_reactants[s,r] + v_products[s,r] <= 1 for s in S for r in R), "ReactanctXorProduct" )

m.addConstrs( ((sum([v_reactants[s,r] for s in S])+sum([v_products[s,r] for s in S]) >= 2) for r in R), "AtLeast2Species" )


m.addConstrs( (sum([v_reactants[s,r] for s in S]) <= 2 for r in R), "AtMost2Reactants" )

m.addConstrs( (sum([v_products[s,r] for s in S]) <= 2 for r in R), "AtMost2Products" )

# m.addConstrs( (sum([v_reactants[s,r] for s in S_site]) <=2 for r in R), "AtMost2SitesInReactants" )

# m.addConstrs( (sum([v_products[s,r] for s in S_site]) <=2 for r in R), "AtMost2SitesInProducts" )

m.addConstrs( (((sum([(v_products[s,r] - v_reactants[s,r]) * Species[s].E for s in S])) <= 0.0) for r in R), "includes only one way")

#m.addConstrs( (((sum([(v_products[s,r] - v_reactants[s,r]) * Species[s].G for s in S])) >= -MaxGibbs) for r in R), "ExcludeHighEnergyStep")


# m.addConstrs( (sum([v_products[s,r] for s in S]) <= 2 for r in R), "NoAllGases" )

# Declare the objective
m.setObjective(1, GRB.MINIMIZE)

# Solve the model
m.update()
m.optimize()
end = time.time()
print("total time taken = {:^.3f} s".format(end-start))

total time taken = 73.262 s


In [65]:
m.SolCount

102815

In [19]:
def is_all_gas(rxn):
    reacts = rxn.strip().split('=')[0]

    if '*' not in reacts and reacts.count('_g') >=1:
        return True

    return False

def is_sur_gas_to_gas(rxn):

    reacts = rxn.strip().split('=')[0]
    prods = rxn.strip().split('=')[1]

    if  reacts.count('*') >=1 and reacts.count('_g') >=1 and prods.count('_g') >=1 :
        return True

    return False

In [20]:
start = time.time()
if (m.SolCount==0):
        toc = timeit.default_timer();
        print('Time elapsed: ', round(toc-tic,1), 'seconds')
        exit();


# Import Numpy
import numpy as np
numExcludedMechanisms=0
n_S = sum(1 for s in S)


indices = np.zeros((m.SolCount,1))
ExcludeMechanism = [False] * m.SolCount
MatchedWith = np.zeros((m.SolCount,1))
Matrix = np.zeros((m.SolCount,len(S)))

for SolInd in range(0, m.SolCount):


        # Set the Gurobi parameter to indicate which solution we are looking at
        m.setParam('SolutionNumber', SolInd)
        indices[SolInd]=SolInd

        countS=0
        coeffRxn1=np.zeros((n_S,1))
        coeffRxn2=np.zeros((n_S,1))
        for r in R:
            for s in S:
                countS=countS+1
                coeffRxn1[countS-1]=(v_products[s,r].xn - v_reactants[s,r].xn)
end = time.time()
print("total time taken = {:^.3f} s".format(end-start))

total time taken = 34.147 s


In [21]:
countUniqueRxns=0

unique_rxns = []
Reactions = []
Energy = []
for SolInd in range(0, m.SolCount):
#     print(SolInd)
    ActualSolInd = int(indices[SolInd])

    # Set the Gurobi parameter to indicate which solution we are looking at
    m.setParam('SolutionNumber', ActualSolInd)

    if (ExcludeMechanism[ActualSolInd] == True):
                numExcludedMechanisms=numExcludedMechanisms+1
    else:
        # Display an update to inform the user which solution we are looking at
        countUniqueRxns=countUniqueRxns+1
#         print('Reaction ' + str(countUniqueRxns))

        for r in R:
            # Build up the reactant string
            ReactantString = ''
            FirstSpecies = True

            for s in S:

                    if (np.round(v_reactants[s,r].xn) == 1):
                            if FirstSpecies == False:
                                    ReactantString = ReactantString + ' + '

                            else:
                                    FirstSpecies = False
                            ReactantString = ReactantString + s

            countSforMatrix=0
            for l in S:
                if (np.round(v_reactants[l,r].xn) == 1):
                    Matrix[SolInd,countSforMatrix] = -1
                countSforMatrix=countSforMatrix+1

            ProductString = ''
            FirstSpecies = True
            for s in S:
                    if (np.round(v_products[s,r].xn) == 1):
                            if FirstSpecies == False:
                                    ProductString = ProductString + ' + '
                            else:
                                    FirstSpecies = False
                            ProductString = ProductString + s
            countSforMatrix=0
            for l in S:
                if (np.round(v_products[l,r].xn) == 1):
                    Matrix[SolInd,countSforMatrix] = 1
                countSforMatrix=countSforMatrix+1

            RxnString = '\t' + ReactantString + ' = ' + '\t' + ProductString
            #Reactions.append(RxnString)
            DelE = sum([(v_products[s,r].xn - v_reactants[s,r].xn) * Species[s].E for s in S])
            Energy.append(DelE)
            DelEString = 'DelE = ' + str(round(DelE,2)) + ' eV'
            
            # Print the full string
            if is_sur_gas_to_gas(RxnString) or is_all_gas(RxnString):
                pass
            else:
                #print(RxnString.ljust(50) + DelEString )
                unique_rxns.append(RxnString)
                Reactions.append(ReactantString + ' = ' + ProductString)

In [24]:
reactions = pd.DataFrame(columns = ['reaction','energy', 'Reactant 1', 'Reactant 2', 'Product 1','Product 2'])
for i in range(len(Reactions)):
    Reactant1 =Reactions[i].split('=')[0].strip("\t").split("+")[0].strip()
    if len(Reactions[i].split('=')[0].strip("\t").split("+")) == 1:
        Reactant2 = ''
    else:
        Reactant2 =Reactions[i].split('=')[0].strip("\t").split("+")[1].strip()
    Product1 =Reactions[i].split('=')[1].strip("\t").split("+")[0].strip()
    if len(Reactions[i].split('=')[1].strip("\t").split("+")) == 1:
        Product2 = ''
    else:
        Product2 =Reactions[i].split('=')[1].strip("\t").split("+")[1].strip()
    smilesr1=species[Reactant1].smiles
    if Reactant2 == '':
        smilesr2="*"
    else:
        smilesr2=species[Reactant2].smiles
    smilesp1=species[Product1].smiles
    if Product2 == '':
        smilesp2 ="*"
    else:
        smilesp2=species[Product2].smiles
    
    E = Energy[i]
    R = Reactions[i]
    reactions.loc[i] = [R,E, smilesr1, smilesr2, smilesp1, smilesp2]    


In [25]:
reactions

,reaction,energy,Reactant 1,Reactant 2,Product 1,Product 2
0,CH3CHCHCH2OH* + O* = CH3CHCHCH2O* + OH*,-0.765991,C/C=C/CO,[O],C/C=C/C[O],[OH]
1,CH3CH2CH2CH2O* + H2* = CH3CH2CH2CH2OH* + H*,-1.075828,CCCC[O],[H][H],CCCCO,[H]
2,CH3CH2CH2CH2O* + H* = CH3CH2CH2CH2OH* + *,-0.499243,CCCC[O],[H],CCCCO,*
3,CH3CH2CHCH2OH* = CH3CH2CH2CH2O*,-0.296733,CC[CH]CO,*,CCCC[O],*
4,CH3CH2CH2CH2O* + O* = CH3CH2CH3* + HCOO*,-1.428058,CCCC[O],[O],CCC,C(=O)[O]
...,...,...,...,...,...,...
102810,CH2CHCHOH* + CH3CHCH2O* = CH2CHCH2OH* + CH3CHCOH*,-0.104807,[CH2][CH][CH]O,C[CH]C[O],[CH2][CH]CO,C[CH][C]O
102811,CH3CH2CHOH* + CH3CHCOH* = CH2CHCH2OH* + CH3CHC...,-0.290095,CC[CH]O,C[CH][C]O,[CH2][CH]CO,C[CH]C[O]
102812,CH3CHCH2OH* + CH3CHCOH* = CH2CHCH2OH* + CH3CHC...,-0.020328,C[CH]CO,C[CH][C]O,[CH2][CH]CO,C[CH]C[O]
102813,CH2CHCHOH* + CH3CH2COH* = CH2CHCH2OH* + CH3CHCOH*,-0.144262,[CH2][CH][CH]O,CC[C]O,[CH2][CH]CO,C[CH][C]O


## Same Reactants

In [26]:
MaxGibbs = float('inf')

# Specify the initial state of the catalyst
# InitialStateSurface = '*'

# Specify the Gibbs free energy change per electron for the complete electrochemical process
# DelGTot = -0.69

n_R = 1


NumSolutionsToFind = 10**9

# Initialize the model
from gurobipy import *
m = Model()

# Specify the parameters related to the "Solution Pool"
m.setParam('OutputFlag', False)
m.setParam('PoolSearchMode',2)  # See the gurobi documentation.  A value of 2 indicates that Gurobi will do a systematic search for the n best solution
m.setParam('PoolSolutions', NumSolutionsToFind)  # Specifies the number of solutions you want the solver to look for
m.setParam('MIPGap',0) # Specifies the tolerance for declaring two objective function values to be the same

# Declare the sets

S = Species.keys()
ListOfSpecies = list(S)
print(ListOfSpecies)

['CCO*', 'CH*', 'CH2*', 'CH2CH*', 'CH2CH2*', 'CH2CHCH2O*', 'CH2CHCH2OH*', 'CH2CHCHO*', 'CH2CHCHOH*', 'CH2CHO*', 'CH2CHOCH2OH*', 'CH2CHOCHO*', 'CH2CHOCHOH*', 'CH2CHOCO*', 'CH2CHOH*', 'CH2CHOHCH2OH*', 'CH2CHOHCHO*', 'CH2CHOHCHOH*', 'CH2CO*', 'CH2O*', 'CH2OH*', 'CH3*', 'CH3CCH*', 'CH3CH2*', 'CH3CH2CH2*', 'CH3CH2CH2CH2O*', 'CH3CH2CH2CH2OH*', 'CH3CH2CH2CHO*', 'CH3CH2CH2CHOH*', 'CH3CH2CH2O*', 'CH3CH2CH2OH*', 'CH3CH2CH3*', 'CH3CH2CHCH2OH*', 'CH3CH2CHCHO*', 'CH3CH2CHO*', 'CH3CH2CHOH*', 'CH3CH2COH*', 'CH3CH2O*', 'CH3CH2OH*', 'CH3CH3*', 'CH3CHCH*', 'CH3CHCH2*', 'CH3CHCH2CH2OH*', 'CH3CHCH2CHO*', 'CH3CHCH2O*', 'CH3CHCH2OH*', 'CH3CHCHCH2O*', 'CH3CHCHCH2OH*', 'CH3CHCHCHO*', 'CH3CHCHCHOH*', 'CH3CHCHO*', 'CH3CHCHOH*', 'CH3CHCOH*', 'CH3CHO*', 'CH3CHOCH2CHO*', 'CH3CHOCHO*', 'CH3CHOCHOH*', 'CH3CHOCO*', 'CH3CHOCOH*', 'CH3CHOH*', 'CH3CHOHCH2CHO*', 'CH3CHOHCHCHO*', 'CH3CHOHCHO*', 'CH3CHOHCHOH*', 'CH3CHOHCO*', 'CH3CHOHCOH*', 'CH3CO*', 'CH3O*', 'CH3OH*', 'CH4*', 'CHCH*', 'CHCH2O*', 'CHCHO*', 'CHCO*', 'CHO*', 

In [27]:
S_site = [s for s in Species.keys() if Species[s].InvolvesSite == True]

S_molecule = [s for s in Species.keys() if ( (Species[s].InvolvesSite == False) and (Species[s].Name != '(H++e-)') )]
R = range(1, n_R+1)
R_subset =  range(1, n_R)


In [28]:
# g = m.addVar(vtype = GRB.CONTINUOUS, name = 'g')

v_products = m.addVars(S, R, vtype = GRB.BINARY, name = 'ProductCoefficients')
v_reactants = m.addVars(S, R, vtype = GRB.BINARY,  name = 'ReactantCoefficients')

In [29]:
# Declare the constraints
start = time.time()
m.addConstrs( (sum([(v_products[s,r] - 2*v_reactants[s,r]) * Species[s].n_H for s in S]) == 0 for r in R), "H_balances"  )

m.addConstrs( (sum([(v_products[s,r] - 2*v_reactants[s,r]) * Species[s].n_O for s in S]) == 0 for r in R), "O_balances"  )

m.addConstrs( (sum([(v_products[s,r] - 2*v_reactants[s,r]) * Species[s].n_C for s in S]) == 0 for r in R), "C_balances"  )

m.addConstrs( (sum([v_products[s,r] - 2*v_reactants[s,r] for s in S_site]) == 0 for r in R), "Site_balances")

G_Values = [s.E for s in Species.values()]
# M = 2*(max(G_Values) - min(G_Values))
# m.addConstrs( ((g >= sum([(v_products[s,r] - 2*v_reactants[s,r]) * Species[s].E for s in S])) for r in R), "Equalities" )

# m.addConstrs([v_reactants[s,r] for s in S])==2 for r in R)
#  or m.addConstrs([v_reactants[s,r] for s in S]==0  for r in R)
# m.addConstrs(v_products[s,r]<=1 for s in S for r in R)
# m.addConstrs( (v_reactants[s,r]*v_products[s,r] == 0 for s in S for  r in R), "ReactanctXorProduct" )

m.addConstrs( (v_reactants[s,r] + v_products[s,r] <= 1 for s in S for  r in R), "ReactanctXorProduct" )

m.addConstrs( ((sum([v_reactants[s,r] for s in S])+sum([v_products[s,r] for s in S]) >= 2) for r in R), "AtLeast2Species" )

m.addConstrs( (sum([v_reactants[s,r] for s in S]) <= 2 for r in R), "AtMost2Reactants" )

m.addConstrs( (sum([v_products[s,r] for s in S]) <= 2 for r in R), "AtMost2Products" )

# m.addConstrs( (sum([v_reactants[s,r] for s in S_site]) <= 2 for r in R), "AtMost2SitesInReactants" )

# m.addConstrs( (sum([v_products[s,r] for s in S_site]) <=2 for r in R), "AtMost2SitesInProducts" )

# m.addConstrs( (((sum([(v_products[s,r] - 2*v_reactants[s,r]) * Species[s].G for s in S])) <= 0.0) for r in R), "includes only one way")


# m.addConstrs( (((sum([(v_products[s,r] - 2*v_reactants[s,r]) * Species[s].G for s in S])) >= -MaxGibbs) for r in R), "ExcludeHighEnergyStep")


# m.addConstrs( (sum([v_products[s,r] for s in S]) <= 2 for r in R), "NoAllGases" )

# Declare the objective
m.setObjective(1, GRB.MINIMIZE)

# Solve the model
m.update()
m.optimize()
end = time.time()
print("total time taken = {:^.3f} s".format(end-start))

total time taken = 0.487 s


In [30]:
m.SolCount

1910

In [31]:
start = time.time()
if (m.SolCount==0):
        toc = timeit.default_timer();
        print('Time elapsed: ', round(toc-tic,1), 'seconds')
        exit();

#tic = timeit.default_timer();

# Import Numpy
import numpy as np
numExcludedMechanisms=0
n_S = sum(1 for s in S)

# Print out the results
# Loop over all the solutions found
# etaList = np.zeros((m.SolCount,1))
# gHNEGSList = np.zeros((m.SolCount,1))
indices = np.zeros((m.SolCount,1))
ExcludeMechanism = [False] * m.SolCount
MatchedWith = np.zeros((m.SolCount,1))
Matrix = np.zeros((m.SolCount,len(S)))

for SolInd in range(0, m.SolCount):


        # Set the Gurobi parameter to indicate which solution we are looking at
        m.setParam('SolutionNumber', SolInd)

#         gmaxNonElectro=-np.inf

#         for r in R:
#                 DelG = sum([(v_products[s,r].xn - v_reactants[s,r].xn) * Species[s].G for s in S])

#                 if (DelG>gmaxNonElectro and round(v_products['(H++e-)',r].xn)==0):
#                         gmaxNonElectro=DelG


#         gHNEGSList[SolInd]=round(gmaxNonElectro,2)
#         gmaxElectro = g.xn
#         eta = gmaxElectro - DelGTot
#         etaList[SolInd] = round(eta,2)
        indices[SolInd]=SolInd

        countS=0
        coeffRxn1=np.zeros((n_S,1))
        coeffRxn2=np.zeros((n_S,1))
        for r in R:
            for s in S:
                countS=countS+1
                coeffRxn1[countS-1]=(v_products[s,r].xn - v_reactants[s,r].xn)

#         for SolInd2 in np.concatenate((range(0,SolInd),range(SolInd+1,m.SolCount))):

#             m.setParam('SolutionNumber', SolInd2)

#             countS=0
#             countZeroS=0
#             countSpeciesRxn=0
#             for r in R:
#                 for s in S:
#                     countS=countS+1
#                     coeffRxn2[countS-1]=(v_products[s,r].xn - v_reactants[s,r].xn)

#                     if (np.round(coeffRxn1[countS-1]+coeffRxn2[countS-1])==0 and np.abs(coeffRxn1[countS-1])>0):
#                         countZeroS=countZeroS+1
#                     if (abs(coeffRxn1[countS-1])>0):
#                                       countSpeciesRxn=countSpeciesRxn+1

#             if (countZeroS == countSpeciesRxn):
#                         MatchedWith[int(np.min((SolInd,SolInd2)))]=int(np.max((SolInd,SolInd2)))
#                         ExcludeMechanism[int(np.max((SolInd,SolInd2)))] = True



# etaGHNEGSList=np.hstack((indices,etaList,gHNEGSList))

# from operator import itemgetter
# sortedEtaGHNEGSList = np.array(sorted(etaGHNEGSList, key=itemgetter(1,2)))

# indices = sortedEtaGHNEGSList[:,0]

end = time.time()
print("total time taken = {:^.3f} s".format(end-start))

total time taken = 0.807 s


In [32]:
countUniqueRxns=0

unique_rxns = []
Reactions = []
Energy = []
reactions_with_same_reactants = pd.DataFrame(columns = ['reaction','energy', 'Reactant 1', 'Reactant 2', 'Product 1','Product 2'])

for SolInd in range(0, m.SolCount):
#     print(SolInd)
    ActualSolInd = int(indices[SolInd])

    # Set the Gurobi parameter to indicate which solution we are looking at
    m.setParam('SolutionNumber', ActualSolInd)

    if (ExcludeMechanism[ActualSolInd] == True):
                numExcludedMechanisms=numExcludedMechanisms+1
    else:
        # Display an update to inform the user which solution we are looking at
        countUniqueRxns=countUniqueRxns+1
#         print('Reaction ' + str(countUniqueRxns))

        for r in R:
            # Build up the reactant string
            ReactantString = ''
            FirstSpecies = True

            for s in S:
                     if (np.round(v_reactants[s,r].xn) == 1):
                      # for s1 in S:
                            if FirstSpecies == False:
                                    ReactantString = ReactantString + ' + '

                            else:
                                    FirstSpecies = False
                            ReactantString = s +'+'+ s

                    # else:
                    #  for s1 in S:
                    #   if (np.round(v_reactants[s1,r].xn) == 2):
                    #           if FirstSpecies == False:
                    #                   ReactantString = ReactantString + ' + '

                    #           else:
                    #                   FirstSpecies = False
                    #           ReactantString = s1+ ' + ' + s1

            countSforMatrix=0
            for l in S:
                if (np.round(v_reactants[l,r].xn) == 1):
                    Matrix[SolInd,countSforMatrix] = -1
                countSforMatrix=countSforMatrix+1

            ProductString = ''
            FirstSpecies = True
            for s in S:
                    if (np.round(v_products[s,r].xn) == 1):
                            if FirstSpecies == False:
                                    ProductString = ProductString + ' + '
                            else:
                                    FirstSpecies = False
                            ProductString = ProductString + s
                    else:
                     for s1 in S:
                      if np.round(v_products[s1,r].xn) == 2:
                            if FirstSpecies == False:
                             ProductString = ProductString + ' + '
                            else:
                                    FirstSpecies = False
                            ProductString =  s1+ ' + ' + s1
            countSforMatrix=0
            for l in S:
                if (np.round(v_products[l,r].xn) == 1):
                    Matrix[SolInd,countSforMatrix] = 1
                countSforMatrix=countSforMatrix+1

            RxnString = '\t' + ReactantString + ' = ' + '\t' + ProductString
            
            DelE = sum([(v_products[s,r].xn - 2*v_reactants[s,r].xn) * Species[s].E for s in S])
            Energy.append(DelE)
            DelGString = 'DelE = ' + str(round(DelE,2)) + ' eV'




            # Print the full string
            if is_sur_gas_to_gas(RxnString) or is_all_gas(RxnString):
                pass
            else:
                #print(RxnString.ljust(50) + DelGString )
                unique_rxns.append(RxnString)
                Reactions.append(ReactantString + ' = ' + ProductString)


In [33]:
reactions_with_same_reactants = pd.DataFrame(columns = ['reaction','energy', 'Reactant 1', 'Reactant 2', 'Product 1','Product 2'])
for i in range(len(Reactions)):
    Reactant1 =Reactions[i].split('=')[0].split("+")[0].strip()
    if len(Reactions[i].split('=')[0].split("+")) == 1:
        Reactant2 = ''
    else:
        Reactant2 =Reactions[i].split('=')[0].split("+")[1].strip()
    Product1 =Reactions[i].split('=')[1].split("+")[0].strip()
    if len(Reactions[i].split('=')[1].split("+")) == 1:
        Product2 = ''
    else:
        Product2 =Reactions[i].split('=')[1].split("+")[1].strip()
    smilesr1=species[Reactant1].smiles
    if Reactant2 == '':
        smilesr2="*"
    else:
        smilesr2=species[Reactant2].smiles
    smilesp1=species[Product1].smiles
    if Product2 == '':
        smilesp2 ="*"
    else:
        smilesp2=species[Product2].smiles
    
    E = Energy[i]
    R= Reactions[i]
    reactions_with_same_reactants.loc[i] = [R,E, smilesr1, smilesr2, smilesp1, smilesp2]    

In [34]:
reactions_with_same_reactants

,reaction,energy,Reactant 1,Reactant 2,Product 1,Product 2
0,H*+H* = H2* + *,0.576585,[H],[H],[H][H],*
1,CH*+CH* = CHCH* + *,-1.866562,[CH],[CH],C#C,*
2,CHOHCH2O*+CHOHCH2O* = CH2CHCOOH* + CH2OHOH*,-0.258740,[O]C[CH]O,[O]C[CH]O,[CH2][CH]C(=O)O,C(O)O
3,CH3CH2OH*+CH3CH2OH* = CH3CH2CH3* + CH2OHOH*,0.790752,CCO,CCO,CCC,C(O)O
4,CH2CHCOH*+CH2CHCOH* = CH2CHOCO* + CH3CHCH*,-0.955764,C=C[C]O,C=C[C]O,[CH2][C@@H]([O])[C][O],C[CH][CH]
...,...,...,...,...,...,...
1905,CHOHCH2O*+CHOHCH2O* = H2COOH* + CH2CHCOHOH*,-1.218333,[O]C[CH]O,[O]C[CH]O,[O]CO,[CH2]C=C(O)O
1906,CHOHCH2O*+CHOHCH2O* = HCOOH* + CH2CHCHOHOH*,-1.477592,[O]C[CH]O,[O]C[CH]O,O=CO,[CH2][CH]C(O)O
1907,CHOHCH2O*+CHOHCH2O* = CH2CHCOHOH* + CHOHOH*,0.805819,[O]C[CH]O,[O]C[CH]O,[CH2]C=C(O)O,[CH](O)O
1908,CH3CH2O*+CH3CH2O* = CH3CH2CH2CH2OH* + O*,0.434154,CC[O],CC[O],CCCCO,[O]


In [35]:
are=pd.concat([reactions, reactions_with_same_reactants])

In [36]:
are=are.to_csv('reactions from ARE for check.csv', index=True)